### Dataset and Task Metadata

In [149]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="kickstarter",
    dataset_year="2025",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Other",
    original_dataset_source_download_link="https://webrobots.io/kickstarter-datasets/",
    download_description="""
There exists data from Kaggle (https://www.kaggle.com/datasets/yashkantharia/kickstarter-campaign).
We use data from the original source (https://webrobots.io/kickstarter-datasets/) and downloaded the newest version at the time of writing.

wget https://s3.amazonaws.com/weruns/forfun/Kickstarter/Kickstarter_2026-01-12T09_37_51_016Z.zip
mkdir -p local-data-warehouse/kickstarter && mv Kickstarter_2026-01-12T09_37_51_016Z.zip local-data-warehouse/kickstarter && mkdir -p local-data-warehouse/kickstarter/data_files && unzip local-data-warehouse/kickstarter/Kickstarter_2026-01-12T09_37_51_016Z.zip -d local-data-warehouse/kickstarter/data_files
""",
    # References
    academic_reference_bibtex=r"""@misc{webrobots2026kickstarter,
  title        = {Kickstarter Datasets},
  author       = {{Web Robots}},
  howpublished = {\url{https://webrobots.io/kickstarter-datasets/}},
  note         = {Accessed: 2026-01-25},
  year         = {2026},
  organization = {Web Robots}
}
""",
    academic_reference_bibtex_key="webrobots2026kickstarter",
    licence="None",
    data_tags=["Non-IID", "Temporal", "Spatial"],
    curation_comments="""
Similar to the Kaggle task, we aim to predict whether a Kickstarter project will be funded successfully.

- The data we used only had 2 samples for 2026, so we stick to data until 2025.
- The original data comes in .csv files per scrape data. We first combine all files into one file.
- We remove all columns that are known only after the outcome and thus could leak information.
- The fx_rate seems to show the conversion rate at the time of crawling the data and not at the time of the project. Thus, if it is used as on Kaggle, the numbers are wrong. We use the "usd_exchange_rate" to transform the goal into USD currency.
- We drop the two entries with "disable_communication" as they point to other issues.
- We drop photo and video based references as we do not include these modalities.
- We decode the category and creator name  from JSON strings into usable columns.
- Note, the crawl does only include blurbs and not the full-text descriptions of the projects. Also some blurb repeat from similar projects or orders.
- We decode the profile blurb, when it exists.
- We decode the location display name from the location JSON string, which can include state and city name.
- The data contains spatial information (city, state, country). We do not decode this but leave it to the pipelines.
- We found 150 rows without location information. We drop these rows as it is unclear which issue caused this and how the rows' data might be affected by this.
- We drop duplicates (20%) which seems to have occurred from multiple scrapes of the same project.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="state",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="state",
    time_on="created_at", # could also be deadline to get more realistic data with projects for which we know the lab at "real" train time. But with our time splits, created_at is fine as well.
)

## Preprocessing

In [187]:
import pandas as pd
import numpy as np
import json

data_dir = dataset_mold.path / "data_files"

# Read and concatenate all CSV files
df = pd.concat(
    (pd.read_csv(csv_file) for csv_file in data_dir.glob("*.csv")),
    ignore_index=True
)
print("Loaded data shape:", df.shape)

# Filter to successful or failed projects
df = df[df["state"].isin(["successful", "failed"])]

# Drop rows with missing location information
df = df[~df["location"].isna()]

# Convert goal to USD
non_usd_mask = df["currency"] != "USD"
df.loc[non_usd_mask, "goal"] = df.loc[non_usd_mask, "goal"] * df.loc[non_usd_mask, "usd_exchange_rate"]

# Use only full country name
df["country"] = df["country_displayable_name"]
df = df.drop(columns=["country_displayable_name"])
# drop disable_communication artifacts
df = df[df["disable_communication"] == False]
df = df.drop(columns=["disable_communication"])

# We do not include images for this benchmark, so we drop the related columns
df = df.drop(columns=["photo", "video"])

# Drop other columns
drop_columns = [
    # Remove target leakage columns
    "backers_count", "converted_pledged_amount", "is_in_post_campaign_pledging_phase",
    "percent_funded", "pledged", "static_usd_rate", "usd_pledged", "usd_type",
    "usd_exchange_rate", "state_changed_at",
    # Remove currency related columns not need anymore
    "currency", "currency_symbol", "currency_trailing_code",
    "current_currency", "fx_rate",
    # all unique
    "is_disliked", "is_launched", "is_liked", "is_starrable",
    # We got the date from created_at
    "id",
    # Just formating of name
    "slug",
    # No relation / not needed
    "source_url", "urls",
]
df = df.drop(columns=drop_columns)

# Decode category
df["main_category"] = df["category"].apply(lambda x: json.loads(x)["name"])
df["sub_category"] = df["category"].apply(lambda x: json.loads(x)["parent_name"] if "parent_name" in json.loads(x) else np.nan)
df = df.drop(columns=["category"])

# Decode profile blurb
def decode_profile(x):
    try:
        data = json.loads(x)
    except json.decoder.JSONDecodeError:
        raw_data = x.split(",")
        raw_data = [e for e in raw_data if e.startswith('"blurb":')]
        assert len(raw_data) == 1
        blurb_entry = raw_data[0]
        blurb_entry = blurb_entry.replace('"blurb":"', "").rstrip('"')
        if "null" in blurb_entry:
            return np.nan
        if blurb_entry == "":
            return np.nan
        return blurb_entry
    res = data["blurb"]
    if res == "":
        return np.nan
    return res

df["profile_blurb"] = df["profile"].apply(decode_profile)
df = df.drop(columns=["profile"])

# Decode creator name
def decode_name(x):
    try:
        data = json.loads(x)
    except json.decoder.JSONDecodeError:
        raw_data = x.split(",")
        raw_data = [e for e in raw_data if e.startswith('"name":')]
        assert len(raw_data) == 1
        name_entry = raw_data[0]
        name_entry = name_entry.replace('"name":"', "").rstrip('"')
        return name_entry
    return data["name"]
df["creator_name"] = df["creator"].apply(decode_name)
df = df.drop(columns=["creator"])

# Decode location
df["location_displayable_name"] = df["location"].apply(lambda x: json.loads(x)["displayable_name"])
df = df.drop(columns=["location"])

# Dtypes
as_string_cols = [
    "blurb",
    "name",
    "creator_name",
    "profile_blurb",
]
as_date_cols = [
    "created_at",
    "launched_at",
    "deadline",
]
as_cat_type = [
    "prelaunch_activated",
    "spotlight",
    "staff_pick",
    "main_category",
    "sub_category",
    "state",
]

for c in as_string_cols:
    nan_mask = df[c].isna()
    df.loc[nan_mask, c] = np.nan
    df[c] = df[c].astype("string")

for c in as_date_cols:
    df[c] = pd.to_datetime(df[c], unit="s")

df[as_cat_type] = df[as_cat_type].astype("category")

# Drop duplicates
df = df.drop_duplicates()

# Drop the 2 samples from 2026
df = df[df["created_at"].dt.year < 2026]

df = df.reset_index(drop=True)

## Data Checks

In [188]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 187,118
Columns: 16

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)

Data quality checks completed.


In [189]:
# Sample Rows
df_head

,blurb,country,created_at,deadline,goal,launched_at,name,prelaunch_activated,spotlight,staff_pick,state,main_category,sub_category,profile_blurb,creator_name,location_displayable_name
0,STRETCH GOAL 17K! If we hit I’ll release 2 extra versions of songs we recorded together that aren’t going on the record to all backers!,the United States,2024-03-26 18:11:47,2024-06-06 15:03:24,15000.0,2024-05-07 15:03:24,"""Night Skin"": Andy Sydow's New Album",True,True,True,successful,Country & Folk,Music,Produced by Anders Osborne at Dockside Studio in Maurice,Andy Sydow,"Nashville, TN"
1,"Captured over 12 years, “Transcending Ellenville” follows three transgender friends as they try to get by in small-town America.",the United States,2025-05-04 20:14:25,2025-07-31 03:59:00,50000.0,2025-07-01 00:57:17,Transcending Ellenville: A Feature Length Documentary,True,True,False,successful,Documentary,Film & Video,<NA>,Gene Fischer,"Ellenville, NY"
2,"Taking my small-batch BBQ rub business to the next level with new flavors, and supporting veterans.",the United States,2018-06-06 03:01:52,2018-07-06 05:35:01,5000.0,2018-06-06 05:35:01,Old Glory BBQ Rubs -- Award Winning,False,False,False,failed,Small Batch,Food,<NA>,Ryan Ledendecker,"Waterloo, IL"
3,"Liquid ChroniK Vodka, This an elegant and harmonious Vodka with strong notes, delivering a spectrum of sensations in 2018.",the United States,2018-04-27 05:21:35,2018-06-10 18:44:12,17500.0,2018-05-11 18:44:12,LIQUID CHRONIK VODKA,False,False,False,failed,Small Batch,Food,<NA>,S.L.Gates,"Las Vegas, NV"
4,A 10-inch vinyl limited edition toy featuring Flint Sparks from the hit comic series Good Boy!,the United States,2023-04-13 21:18:26,2023-06-24 16:51:58,9499.0,2023-05-25 16:51:58,Good Boy Vinyl Collectible,True,True,True,successful,Toys,Design,<NA>,Garrett Gunn,"Saginaw, MI"


In [190]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,sub_category,category,4238,2.26,15,"Film & Video, Music, Publishing, Technology, Art, Food, Games, Fashion, Design, Crafts"
1,prelaunch_activated,category,0,0.00,2,"False, True"
2,spotlight,category,0,0.00,2,"True, False"
3,staff_pick,category,0,0.00,2,"False, True"
4,state,category,0,0.00,2,"successful, failed"
5,main_category,category,0,0.00,161,"Tabletop Games, Web, Product Design, Anthologies, Comedy, Documentary, Apparel, Comic Books, Jazz, Food Trucks"
6,created_at,datetime64[ns],0,0.00,187058,"2018-01-10 21:11:35, 2015-07-02 19:42:56, 2025-05-12 08:29:20, 2022-01-30 22:35:57, 2025-04-06 14:58:26, 2024-11-01 13:29:25, 2021-12-29 01:24:32, 2025-10-11 16:58:20, 2015-01-17 18:01:26, 2015-02-26 19:24:02"
7,deadline,datetime64[ns],0,0.00,177545,"2025-11-01 03:59:00, 2024-11-01 03:59:00, 2024-09-01 03:59:00, 2025-11-01 06:59:00, 2021-11-01 03:59:00, 2015-01-01 04:59:00, 2014-05-30 21:00:00, 2024-05-01 03:59:00, 2016-04-01 03:59:00, 2025-10-01 03:59:00"
8,launched_at,datetime64[ns],0,0.00,186897,"2025-07-01 14:00:01, 2024-03-07 16:36:02, 2024-09-17 15:00:08, 2025-10-07 14:00:01, 2025-02-25 17:59:17, 2022-05-18 17:00:14, 2018-02-19 17:02:12, 2014-08-13 14:33:29, 2015-01-15 21:20:36, 2025-10-07 13:00:02"
9,goal,float64,0,0.00,66156,"5000.0, 10000.0, 1000.0, 3000.0, 2000.0, 500.0, 15000.0, 2500.0, 20000.0, 1500.0"


In [191]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
goal,187118.0,34317.314037,1.005790e+06,0.01,135734937.0


In [192]:
# Categorical Feature Statistics
cat_stats

value  \
column                    rank                                                                        
blurb                     1     3D Printable STL Files, Terrain for Tabletop, Miniature, Role Pl...   
                          2     High-quality STL Files, 3D Pin-Up Printable Figures for Miniatur...   
                          3     A high-quality Figurines, STL file, 3D printable Presupported Minis   
                          4            Pre-Supported High Quality 3D Printable Figures (STL Files).   
                          5     3D printable miniatures of female characters 75 mm and 32 mm for...   
country                   1                                                       the United States   
                          2                                                      the United Kingdom   
                          3                                                                  Canada   
                          4                                                               Australia   
                          5                                                                 Germany   
created_at                1                                                     2018-01-10 21:11:35   
                          2                                                     2015-07-02 19:42:56   
                          3                                                     2025-05-12 08:29:20   
                          4                                                     2022-01-30 22:35:57   
                          5                                                     2025-04-06 14:58:26   
creator_name              1                                                    Microcosm Publishing   
                          2                                                                    Juan   
                          3                                                            Mike Hoffman   
                          4                                                                   David   
                          5                                                                  Daniel   
deadline                  1                                                     2025-11-01 03:59:00   
                          2                                                     2024-11-01 03:59:00   
                          3                                                     2024-09-01 03:59:00   
                          4                                                     2025-11-01 06:59:00   
                          5                                                     2021-11-01 03:59:00   
launched_at               1                                                     2025-07-01 14:00:01   
                          2                                                     2024-03-07 16:36:02   
                          3                                                     2024-09-17 15:00:08   
                          4                                                     2025-10-07 14:00:01   
                          5                                                     2025-02-25 17:59:17   
location_displayable_name 1                                                         Los Angeles, CA   
                          2                                                              London, UK   
                          3                                                            New York, NY   
                          4                                                             Chicago, IL   
                          5                                                            Brooklyn, NY   
main_category             1                                                          Tabletop Games   
                          2                                                                     Web   
                          3                                                  

In [193]:
# Target Distribution
target_df

,count,pct
state,,
successful,117148,62.61
failed,69970,37.39


## Task Curation

In [194]:
# Filter to year 2025

# Create a year-month column for grouping
df["year"] = df[task_mold.time_on].dt.to_period("Y")
monthly_totals = (
    df
    .groupby("year")
    .size()
    .rename("total_samples")
)
monthly_class_counts = (
    df
    .groupby(["year", task_mold.target_column_name])
    .size()
    .unstack(fill_value=0)
)
result = monthly_class_counts.join(monthly_totals)
result = result.sort_index()
result.index = result.index.to_timestamp()
result

/tmp/ipykernel_98925/543726850.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(["year", task_mold.target_column_name])


,failed,successful,total_samples
year,,,
2009-01-01,5,118,123
2010-01-01,83,784,867
2011-01-01,290,2402,2692
2012-01-01,645,4644,5289
2013-01-01,950,5058,6008
2014-01-01,7590,8578,16168
2015-01-01,9856,8750,18606
2016-01-01,6941,7344,14285
2017-01-01,6374,7822,14196


In [195]:
from data_foundry.schema import PredictiveMLSplitsMetadata

# Sort by time
df = df.sort_values(by=task_mold.time_on).reset_index(drop=True)

splits = {}

test_years = [
    "2023",
    "2024",
    "2025",
]
for i, month in enumerate(test_years):
    ref_date  = pd.Timestamp(month)
    train_index = df[
        df[task_mold.time_on] < ref_date
    ].index
    test_index = df[
        (df[task_mold.time_on].dt.year == ref_date.year)
    ].index
    splits[i] = {
        0: (train_index.tolist(), test_index.tolist())
    }

for s in splits:
    train_index, test_index = splits[s][0]
    print(f"Split {s}: Train size: {len(train_index)}, Test size: {len(test_index)}")
    assert df[task_mold.time_on].iloc[train_index].max() < df[task_mold.time_on].iloc[test_index].min()

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="""We try to create splits that simulate a model deployed to solve the task.

The official data is updated monthly but has not enough data per month to create large enough test splits. We opt for simulating a model that is refit every year to obtain a robust test set.

 We could simulate a model that is refitted every month, but this would need many splits. Moreover, data from just one month is not enough to create a robust test set. We instead simulate a model that is refit every year. This introduces the unrealistic downside of data shift across a month that would not exist in a real-world model. We create 3 test splits by 2023, 2024, and 2025 as test year. For each test split, we use all data before the test month as training data.
""",
    splits=splits,
)

Split 0: Train size: 131836, Test size: 13056
Split 1: Train size: 144892, Test size: 21400
Split 2: Train size: 166292, Test size: 20826


## Export

In [196]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019bf612-ff21-7d34-854f-59808ec61863
f840dcc3446a19d525341a1f34ab8cf03cbb52bc08870e6f7b59c9019f33ddad
